# Example 10.3 / 2: Balance Materials in an Input-Output Model

Companion notebook for **10.3 Columnwise Formulations** in the AMPL lab Documentation, section "Balance Materials in an Input-Output Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Balance Materials in an Input-Output Model

An input-output model is organized by activities. Each activity `j` runs at level `Run[j]`, and
`io[i,j]` is the amount of material `i` that one unit of the activity produces (positive) or
consumes (negative). Every material must balance:

$$\sum_{j \in ACT} io_{ij}\,Run_j - Sell_i = 0 \quad \text{for every } i \in MAT,$$

with $Sell_i$ present only for finished materials. Written by rows, the right side needs
`if i in MATF then Sell[i] else 0`. Written by columns, each activity and each sale brings its own
coefficients, and a coefficient that no `coeff` phrase names is zero. Here an indexed phrase
`coeff {i in MAT} Balance[i] io[i,j]` adds one coefficient per material, and `Sell[i]` has the
single coefficient -1 in its own balance row:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MAT;  # materials
set ACT;  # purchase and production activities
param io {MAT,ACT} default 0;  # output (+) or input (-) of a material per unit of activity
set MATF within MAT;  # finished materials that can be sold
param revenue {MATF} >= 0;  # revenue per unit sold
param sell_min {MATF} >= 0;  # minimum sales
param sell_max {i in MATF} >= sell_min[i];  # maximum sales
param cost {ACT} >= 0;  # cost per unit of activity
param act_min {ACT} >= 0;  # minimum activity level
param act_max {j in ACT} >= act_min[j];  # maximum activity level

maximize Net_Profit;  # terms come from the Run and Sell columns
subject to Balance {i in MAT}: to_come = 0;  # net output of each material is zero

var Run {j in ACT} >= act_min[j], <= act_max[j],  # activity level
    obj Net_Profit -cost[j],
    coeff {i in MAT} Balance[i] io[i,j];
var Sell {i in MATF} >= sell_min[i], <= sell_max[i],  # sales of a finished material
    obj Net_Profit revenue[i],
    coeff Balance[i] -1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["MAT"] = ["ore", "steel", "sheet"]
ampl.set["ACT"] = ["buy_ore", "smelt", "roll"]
ampl.param["io"] = {
    ("ore", "buy_ore"): 1, ("ore", "smelt"): -2,
    ("steel", "smelt"): 1, ("steel", "roll"): -1,
    ("sheet", "roll"): 1,
}
ampl.set["MATF"] = ["steel", "sheet"]
ampl.param["revenue"] = {"steel": 70, "sheet": 100}
ampl.param["sell_min"] = {"steel": 0, "sheet": 0}
ampl.param["sell_max"] = {"steel": 50, "sheet": 40}
ampl.param["cost"] = {"buy_ore": 10, "smelt": 30, "roll": 20}
ampl.param["act_min"] = {"buy_ore": 0, "smelt": 0, "roll": 0}
ampl.param["act_max"] = {"buy_ore": 100, "smelt": 60, "roll": 30}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Net_Profit"].value())
print(ampl.var["Run"].to_dict())
print(ampl.var["Sell"].to_dict())

HiGHS 1.14.0: optimal solution; objective 1300
0 simplex iterations
0 barrier iterations
solved 1300.0
{'buy_ore': 100, 'roll': 30, 'smelt': 50}
{'sheet': 30, 'steel': 20}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The `expand` command prints each balance row as
the columns filled it:

In [6]:
%%writefile data.dat
set MAT := ore steel sheet;
set ACT := buy_ore smelt roll;
param io:   buy_ore  smelt  roll :=
  ore         1       -2      0
  steel       0        1     -1
  sheet       0        0      1 ;
set MATF := steel sheet;
param: revenue sell_min sell_max :=
  steel    70     0     50
  sheet   100     0     40 ;
param: cost act_min act_max :=
  buy_ore  10    0    100
  smelt    30    0     60
  roll     20    0     30 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Run, Sell;
expand Balance;

HiGHS 1.14.0: optimal solution; objective 1300
0 simplex iterations
0 barrier iterations
:         Run Sell    :=
buy_ore   100    .
roll       30    .
sheet      .    30
smelt      50    .
steel      .    20
;

subject to Balance['ore']:
	Run['buy_ore'] - 2*Run['smelt'] = 0;

subject to Balance['steel']:
	Run['smelt'] - Run['roll'] - Sell['steel'] = 0;

subject to Balance['sheet']:
	Run['roll'] - Sell['sheet'] = 0;

